# Saved residual64 inference
Step 2 selected model:64 tokens/vector,context512. Validation999/1000 short and202/206 packed exact. Not universal losslessness. No training in this notebook.


In [1]:
from pathlib import Path
import os, sys, json
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT / 'src').is_dir(), 'Open this notebook from its repository'
os.chdir(ROOT)
if sys.platform == 'win32':
    import ctypes
    k=ctypes.WinDLL('kernel32');k.GetCurrentProcess.restype=ctypes.c_void_p
    k.SetProcessAffinityMask.argtypes=(ctypes.c_void_p,ctypes.c_size_t)
    assert k.SetProcessAffinityMask(k.GetCurrentProcess(),1<<16)
os.environ['TOKENIZERS_PARALLELISM']='false'
for key in ('OMP_NUM_THREADS','MKL_NUM_THREADS','OPENBLAS_NUM_THREADS'): os.environ[key]='1'
sys.path.insert(0,str(ROOT/'src'))
import torch
torch.set_num_threads(1)


In [2]:
from models.position_compressor.codec import Codec
config=json.loads((ROOT/'src/config/position_compressor.json').read_text())
checkpoint=Path(config['checkpoint'])
if not checkpoint.is_absolute(): checkpoint=ROOT/checkpoint
assert checkpoint.is_file(), 'Saved weights are local and are not included in Git'
codec=Codec.load(checkpoint, device='cuda' if torch.cuda.is_available() else 'cpu')
print(codec.model.config)


Config(vocab_size=4096, width=256, heads=4, encoder_layers=4, decoder_layers=1, max_tokens=512, span=64, hidden=1024, ffn='branch_sigmoid_v1', position_encoding='rope_v1', rope_base=10000.0, pad_id=0, bos_id=1, eos_id=2, variant='depth_route', code_features=125)


In [3]:
text='Invoice 17019: 17.09, not 19.0y79534rejpowf[s\d]c7.(ASDAS'
result=codec.reconstruct(text)
result


<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\awais\AppData\Local\Temp\ipykernel_15612\2874073955.py:1: SyntaxWarning: invalid escape sequence '\d'
  text='Invoice 17019: 17.09, not 19.0y79534rejpowf[s\d]c7.(ASDAS'


{'input': 'Invoice 17019: 17.09, not 19.0y79534rejpowf[s\\d]c7.(ASDAS',
 'output': 'Invoice 17019: 17.09, not 19.0y79534rejpowf[s\\d]c7.(ASDAS',
 'exact_match': True,
 'input_tokens': 40,
 'output_vectors': 1,
 'features_per_vector': 256,
 'output_features': 256,
 'tokens_per_vector': 40.0,
 'vector_bytes': 512,
 'length_metadata_bytes': 8,
 'identity_metadata_bytes': 198,
 'original_utf8_bytes': 57,
 'token_id_bytes_int32': 160,
 'chunks': 1,
 'precision': 'bf16'}